# Phase 3 — End-to-End Three-Head Classifier Training

## Overview
Phase 3 is the first time all layers of the system run together. The MentalRoBERTa backbone (trained in Phase 1), the Graph Attention Network (trained in Phase 2), and three classification heads are now connected into a single pipeline and trained jointly.

The three heads are:
- **Emotion Head** — multi-label sigmoid, predicts 28 GoEmotions categories
- **MH Category Head** — binary softmax, predicts depressed vs. non-depressed
- **Severity Head** — regression, predicts PHQ-8 equivalent score (frozen until DAIC-WOZ data access is obtained)

This cell installs required libraries and confirms the GPU is active. Both Phase 1 and Phase 2 checkpoints are verified to be accessible before any training begins.

In [ ]:
!pip install -q transformers datasets scikit-learn accelerate
!pip install -q torch-geometric -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__.split('+')[0])")+cu121.html

import os, torch, numpy as np, pandas as pd
from torch import nn
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# confirm inputs visible
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if any(x in f.lower() for x in ["gat", "mental", "phase"]):
            print(os.path.join(root, f))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.5 MB/s eta 0:00:00
Device: cuda
/kaggle/input/notebooks/shashwatkashyap12221/phase-2/gat_phase2_final.pt
/kaggle/input/notebooks/shashwatkashyap12221/phase-2/gat_phase2_best.pt


## Step 1 — Load Dataset

GoEmotions is used as the primary training dataset for Phase 3. It is a large-scale multi-label emotion dataset released by Google, containing 211,225 Reddit comments annotated across 28 emotion categories by human raters.

The three raw CSV files (`goemotions_1`, `goemotions_2`, `goemotions_3`) are combined into a single dataframe. The output confirms **211,225 rows and 37 columns**, which includes metadata columns (author, subreddit, rater_id etc.) alongside the 28 binary emotion label columns.

> Note: DepressionEmo was the originally planned dataset for this phase. It was unavailable via HuggingFace at the time of training, so GoEmotions is used instead, which is an equally valid source of fine-grained emotion signal for training the emotion head.

In [ ]:
import glob, pandas as pd

go_files = sorted(glob.glob("/kaggle/input/**/goemotions_*.csv", recursive=True))
print(go_files)

go_df = pd.concat([pd.read_csv(f) for f in go_files], ignore_index=True)
print(go_df.shape)
print(go_df.columns.tolist())

['/kaggle/input/datasets/shashwatkashyap12221/goemotions/goemotions_1.csv', '/kaggle/input/datasets/shashwatkashyap12221/goemotions/goemotions_2.csv', '/kaggle/input/datasets/shashwatkashyap12221/goemotions/goemotions_3.csv']
(211225, 37)
['text', 'id', 'author', 'subreddit', 'link_id', 'parent_id', 'created_utc', 'rater_id', 'example_very_unclear', 'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


## Step 2 — Clean and Deduplicate

The raw GoEmotions data has **one row per annotator per comment** — meaning the same comment can appear multiple times rated by different people. Before training, this is collapsed to one row per unique comment using `groupby("id")`.

The aggregation strategy takes the **max label value** across all raters for each emotion — meaning if any rater marked an emotion as present (1), the final label for that comment is 1. This is the standard approach used in the original GoEmotions paper and captures the union of rater judgments.

After cleaning and deduplication: **58,009 unique comments** remain, each with 28 binary emotion labels — matching the official GoEmotions paper's reported dataset size exactly.

In [ ]:
EMOTION_COLS = ['admiration','amusement','anger','annoyance','approval','caring',
                'confusion','curiosity','desire','disappointment','disapproval',
                'disgust','embarrassment','excitement','fear','gratitude','grief',
                'joy','love','nervousness','optimism','pride','realization',
                'relief','remorse','sadness','surprise','neutral']

go_clean = go_df[go_df["example_very_unclear"] == False].copy()
agg = go_clean.groupby("id").agg(
    {**{"text":"first"}, **{c:"max" for c in EMOTION_COLS}}
).reset_index()

print(agg.shape)
print(f"Emotion columns: {len(EMOTION_COLS)}")

(58009, 30)
Emotion columns: 28


## Step 3 — Train/Validation/Test Split and Checkpoint Discovery

The 58,009 cleaned comments are split into:
- **Train: 49,307** (85%)
- **Validation: 4,351** (7.5%) — used during training to track F1 and save best model
- **Test: 4,351** (7.5%) — held out, only evaluated after all training is complete

`random_state=42` ensures the split is reproducible across sessions.

Both Phase 1 and Phase 2 checkpoints are located dynamically using `glob` so the code works regardless of the exact Kaggle input folder path.

In [ ]:
import glob
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(agg, test_size=0.15, random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42)
print(train_df.shape, val_df.shape, test_df.shape)

# find Phase 1 checkpoint
phase1_paths = glob.glob("/kaggle/input/**/mentalroberta_phase1_final", recursive=True)
PHASE1_PATH = phase1_paths[0] if phase1_paths else None
print("Phase 1 path:", PHASE1_PATH)

# find Phase 2 checkpoint
phase2_paths = glob.glob("/kaggle/input/**/gat_phase2_best.pt", recursive=True)
PHASE2_PATH = phase2_paths[0] if phase2_paths else None
print("Phase 2 path:", PHASE2_PATH)

(49307, 30) (4351, 30) (4351, 30)
Phase 1 path: /kaggle/input/notebooks/shashwatkashyap12221/phase-1/mentalroberta_phase1_final
Phase 2 path: /kaggle/input/notebooks/shashwatkashyap12221/phase-2/gat_phase2_best.pt


## Step 4 — Tokenizer and Dataset Class

The MentalRoBERTa tokenizer from Phase 1 is reused here. Each text is tokenized to a fixed length of 128 tokens with padding and truncation.

The `GoEmotionsDataset` class returns three items per sample:
- `input_ids` — tokenized text as integer IDs
- `attention_mask` — 1 for real tokens, 0 for padding
- `labels` — a 28-dimensional float vector of binary emotion labels

Batch sizes of 16 (train) and 32 (val/test) are used. The larger validation batch size speeds up evaluation since no gradients are computed there.

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader
import torch, numpy as np

tokenizer = AutoTokenizer.from_pretrained(PHASE1_PATH)

class GoEmotionsDataset(Dataset):
    def __init__(self, df, max_len=128):
        self.texts  = df["text"].tolist()
        self.labels = df[EMOTION_COLS].values.astype(np.float32)
        self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], truncation=True,
                        padding="max_length", max_length=self.max_len,
                        return_tensors="pt")
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx])
        }

train_ds = GoEmotionsDataset(train_df)
val_ds   = GoEmotionsDataset(val_df)
test_ds  = GoEmotionsDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32)
test_loader  = DataLoader(test_ds,  batch_size=32)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Train: 49307 | Val: 4351 | Test: 4351


## Step 5 — Phase 3 Model Architecture

The `Phase3Model` is a four-layer system matching the project's architecture exactly:

**Layer 1 — MentalRoBERTa backbone**: loaded from the Phase 1 checkpoint, which was fine-tuned on both Dreaddit and GoEmotions. Produces 768-dimensional token embeddings.

**Layer 2 — GAT (Graph Attention Network)**: two GATConv layers matching Phase 2's exact architecture:
- `gat1`: input 768 → 4 attention heads × 256 = 1024 output
- `gat2`: input 1024 → 2 attention heads → 256 output (concatenation disabled, so final dim = 256)

Token embeddings are treated as graph nodes, with sequential edges connecting token i → token i+1. The `_edge_cache` dictionary pre-builds edge tensors for each unique (batch_size, seq_len) combination so they are not recomputed on every forward pass.

**Layer 3 — Three classification heads**, all taking the 256-dim GAT output:
- `emotion_head`: 256 → 128 → 28 (multi-label)
- `mh_head`: 256 → 64 → 2 (binary classification)
- `severity_head`: 256 → 64 → 1 (regression, **frozen** — all parameters set to `requires_grad=False`)

**Load report warnings** (UNEXPECTED/MISSING) are expected here — the Phase 1 checkpoint was saved as a sequence classifier with a 2-class head, but we are loading it into a base `AutoModel`. HuggingFace drops the old classifier weights (UNEXPECTED) and initialises the pooler fresh (MISSING), which is the correct behaviour.

In [ ]:
from torch_geometric.nn import GATConv
from torch import nn
from transformers import AutoModel

class Phase3Model(nn.Module):
    def __init__(self, backbone_path, num_emotions=28):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(backbone_path)
        self.gat1 = GATConv(768,  256, heads=4, concat=True)
        self.gat2 = GATConv(1024, 256, heads=2, concat=False)

        self.emotion_head = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, num_emotions)
        )
        self.mh_head = nn.Sequential(
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 2)
        )
        self.severity_head = nn.Sequential(
            nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid()
        )
        for p in self.severity_head.parameters():
            p.requires_grad = False

        # pre-build edges once for max sequence length
        self._edge_cache = {}

    def _get_edges(self, B, S):
        key = (B, S)
        if key not in self._edge_cache:
            src, dst = [], []
            for b in range(B):
                offset = b * S
                s = torch.arange(offset, offset + S - 1)
                d = torch.arange(offset + 1, offset + S)
                src.append(s); dst.append(d)
            self._edge_cache[key] = torch.stack(
                [torch.cat(src), torch.cat(dst)]
            )
        return self._edge_cache[key].to(device)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids,
                            attention_mask=attention_mask)
        B, S, H = out.last_hidden_state.shape
        x = out.last_hidden_state.reshape(B * S, H)

        edge_index = self._get_edges(B, S)

        x = torch.relu(self.gat1(x, edge_index))
        x = torch.relu(self.gat2(x, edge_index))

        # CLS token per sample
        cls_idx = torch.arange(B, device=x.device) * S
        graph_vec = x[cls_idx]

        return (self.emotion_head(graph_vec),
                self.mh_head(graph_vec),
                self.severity_head(graph_vec).squeeze(-1))

model = Phase3Model(PHASE1_PATH, num_emotions=len(EMOTION_COLS)).to(device)
print("Optimised model ready")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: /kaggle/input/notebooks/shashwatkashyap12221/phase-1/mentalroberta_phase1_final
Key                        | Status     | 
---------------------------+------------+-
classifier.dense.bias      | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
classifier.dense.weight    | UNEXPECTED | 
classifier.out_proj.weight | UNEXPECTED | 
pooler.dense.weight        | MISSING    | 
pooler.dense.bias          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Optimised model ready


## Step 6 — Transfer Phase 2 GAT Weights

The GAT layers trained in Phase 2 on PRIMATE (DSM-5 symptom classification) are loaded into the model. By inspecting the Phase 2 checkpoint's key names and shapes, 8 out of 10 layers match exactly:

- `gat1.att_src`, `gat1.att_dst`, `gat1.bias`, `gat1.lin.weight`
- `gat2.att_src`, `gat2.att_dst`, `gat2.bias`, `gat2.lin.weight`

The 2 unmatched layers are the Phase 2 `classifier.weight` and `classifier.bias` — that was a 9-class PRIMATE symptom head which is intentionally replaced by our three-head design. This is expected and correct.

**Why this matters**: the GAT layers already know which grammatical relationships (negation, symptom modifiers) carry clinical signal from Phase 2 training. Phase 3 fine-tunes this knowledge further in the context of emotion classification rather than starting from random weights.

In [ ]:
if PHASE2_PATH:
    gat_ckpt   = torch.load(PHASE2_PATH, map_location=device)
    model_dict = model.state_dict()
    matched    = {k: v for k, v in gat_ckpt.items()
                  if k in model_dict and model_dict[k].shape == v.shape}
    model_dict.update(matched)
    model.load_state_dict(model_dict)
    print(f"Phase 2 weights loaded: {len(matched)}/{len(gat_ckpt)} layers matched")
else:
    print("No Phase 2 checkpoint found — continuing from Phase 1 only")

Phase 2 weights loaded: 8/10 layers matched


## Step 7 — Loss Functions and Differential Learning Rates

**Two loss functions** are used with a weighted combination:
- `BCEWithLogitsLoss` (Binary Cross Entropy) for the emotion head — appropriate for multi-label problems where multiple emotions can be present simultaneously
- `CrossEntropyLoss` for the MH category head — appropriate for mutually exclusive binary classification

The combined loss is: `total_loss = 0.6 × BCE_emotion + 0.4 × CE_mh`

The higher weight on emotion (0.6) reflects that GoEmotions is the primary training signal here. The MH label is derived synthetically — a comment is labelled "depressed" if any emotion label is active — so it gets a lower weight.

**Differential learning rates** are applied — a standard practice when fine-tuning pretrained models:
- Backbone (MentalRoBERTa): `lr = 2e-5` — low rate to preserve pretrained knowledge
- GAT layers and classifier heads: `lr = 5e-5` — higher rate to adapt these layers faster

The severity head is excluded from the optimizer entirely since its parameters are frozen.

In [ ]:
bce = nn.BCEWithLogitsLoss()
ce  = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(), "lr": 2e-5},
    {"params": model.gat1.parameters(),     "lr": 5e-5},
    {"params": model.gat2.parameters(),     "lr": 5e-5},
    {"params": model.emotion_head.parameters(), "lr": 5e-5},
    {"params": model.mh_head.parameters(),      "lr": 5e-5},
], weight_decay=0.01)

print("Optimizer ready")

Optimizer ready


## Step 8 — Training Loop (3 Epochs)

The model is trained for 3 epochs over 49,307 training samples with a batch size of 16, giving **3,082 gradient update steps per epoch**.

After each epoch, the model switches to `eval()` mode and runs on the validation set. Emotion probabilities are obtained by applying `sigmoid` to the raw logits (since `BCEWithLogitsLoss` works on raw logits during training), then thresholded at **0.3** to produce binary predictions. A threshold of 0.3 rather than 0.5 is used because GoEmotions labels are sparse — most emotions are absent, so a lower threshold improves recall on rare emotions without hurting precision too much.

**Results:**
| Epoch | Avg Train Loss | Val F1-macro |
|-------|---------------|--------------|
| 1 | 0.1326 | 0.4573 |
| 2 | 0.1144 | 0.4885 |
| 3 | 0.1067 | **0.5008** ← best saved |

Loss decreases consistently across all three epochs, confirming stable learning. F1 improves each epoch, with the model crossing **0.50 by epoch 3** — consistent with published BERT-based baselines on GoEmotions (range: 0.46–0.54).

In [ ]:
!pip install -q tqdm
from tqdm import tqdm
from sklearn.metrics import f1_score
import numpy as np

EPOCHS = 3
best_f1 = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)

    for batch in loop:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labs = batch["labels"].to(device)

        optimizer.zero_grad()
        emo_logits, mh_logits, _ = model(ids, mask)
        mh_labs = (labs.sum(dim=1) > 0).long()
        loss = 0.6 * bce(emo_logits, labs) + 0.4 * ce(mh_logits, mh_labs)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        # update progress bar with current loss
        loop.set_postfix(loss=f"{loss.item():.4f}")

    # validation
    model.eval()
    all_preds, all_labs = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating", leave=False):
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            emo_logits, _, _ = model(ids, mask)
            probs = torch.sigmoid(emo_logits).cpu().numpy()
            all_preds.append((probs >= 0.3).astype(int))
            all_labs.append(batch["labels"].numpy())

    f1 = f1_score(np.vstack(all_labs), np.vstack(all_preds),
                  average="macro", zero_division=0)
    print(f"\nEpoch {epoch+1}/{EPOCHS} | Avg Loss: {total_loss/len(train_loader):.4f} | Val F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "/kaggle/working/phase3_best.pt")
        print("  → best checkpoint saved")

Epoch 1/3: 100%|██████████| 3082/3082 [20:04<00:00,  2.56it/s, loss=0.0824]



Epoch 1/3 | Avg Loss: 0.1326 | Val F1: 0.4573
  → best checkpoint saved


Epoch 2/3: 100%|██████████| 3082/3082 [20:09<00:00,  2.55it/s, loss=0.1178]



Epoch 2/3 | Avg Loss: 0.1144 | Val F1: 0.4885
  → best checkpoint saved


Epoch 3/3: 100%|██████████| 3082/3082 [20:10<00:00,  2.55it/s, loss=0.1210]



Epoch 3/3 | Avg Loss: 0.1067 | Val F1: 0.5008
  → best checkpoint saved


## Step 9 — Final Test Evaluation

The best checkpoint (epoch 3, Val F1 = 0.5008) is loaded and evaluated on the **held-out test set of 4,351 samples**.

**Key observations from the classification report:**

**Strong performing emotions** (F1 > 0.70): `admiration (0.70)`, `amusement (0.78)`, `gratitude (0.76)`, `love (0.78)`, `neutral (0.78)`, `curiosity (0.70)` — these are high-frequency, clearly expressed emotions with sufficient training examples.

**Moderate performers** (F1 0.44–0.60): `anger`, `annoyance`, `approval`, `confusion`, `disapproval`, `fear`, `remorse`, `sadness`, `surprise` — these are semantically overlapping emotions that are harder to distinguish even for human raters.

**Low performers** (F1 < 0.30): `grief (0.08)`, `pride (0.02)`, `relief (0.16)`, `nervousness (0.28)` — these are **rare classes** with very few training examples (49, 99, 92, 119 samples respectively in the test set). The model correctly identifies them with high precision (grief: 1.00, pride: 1.00) but almost never predicts them, resulting in near-zero recall. This is a known class imbalance problem in GoEmotions, not a model failure.

**Overall test metrics:**
- Macro F1: **0.49** (average across all 28 classes including rare ones)
- Micro F1: **0.59** (weighted by class frequency — more representative of real performance)
- Samples F1: **0.61** (per-sample average — best reflects multi-label performance)

**Phase 3 status**: Emotion head and MH category head trained. Severity head remains frozen pending DAIC-WOZ data access. Final checkpoints saved to `/kaggle/working/phase3_best.pt`, `phase3_final.pt`, and `phase3_tokenizer/`.

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

model.load_state_dict(torch.load("/kaggle/working/phase3_best.pt"))
model.eval()

all_preds, all_labs = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        emo_logits, _, _ = model(ids, mask)
        probs = torch.sigmoid(emo_logits).cpu().numpy()
        all_preds.append((probs >= 0.3).astype(int))
        all_labs.append(batch["labels"].numpy())

print("=== PHASE 3 — EMOTION HEAD TEST RESULTS ===")
print(classification_report(
    np.vstack(all_labs), np.vstack(all_preds),
    target_names=EMOTION_COLS, zero_division=0
))

# save final
torch.save(model.state_dict(), "/kaggle/working/phase3_final.pt")
tokenizer.save_pretrained("/kaggle/working/phase3_tokenizer")
print("\nPhase 3 complete. Checkpoints saved.")

Testing: 100%|██████████| 136/136 [00:31<00:00,  4.33it/s]


=== PHASE 3 — EMOTION HEAD TEST RESULTS ===
                precision    recall  f1-score   support

    admiration       0.70      0.71      0.70       764
     amusement       0.85      0.71      0.78       408
         anger       0.52      0.54      0.53       417
     annoyance       0.48      0.54      0.51       712
      approval       0.51      0.48      0.50       990
        caring       0.46      0.50      0.48       343
     confusion       0.52      0.64      0.57       396
     curiosity       0.63      0.79      0.70       432
        desire       0.50      0.44      0.46       181
disappointment       0.41      0.47      0.44       490
   disapproval       0.47      0.58      0.52       615
       disgust       0.47      0.47      0.47       303
 embarrassment       0.42      0.23      0.30       132
    excitement       0.39      0.39      0.39       318
          fear       0.54      0.58      0.56       167
     gratitude       0.78      0.73      0.76       385
   